# Train the shipped model + its two Advanced-selector alternates

This notebook fits and saves the three models the site can show:

| key | notes/model_history.md row | features | why it's here |
|---|---|---|---|
| `shipped` | Shipped model | 11 | **default** everywhere — best COMBINED score (0.827) |
| `final5swing` | Feature-set history, row 5 | 8 | second-best (0.638), "won 28/30 CV seeds" at that stage |
| `goalsonly` | Feature-set history, row 0 | 1 | the plain baseline (goals only) |

`shipped`'s 11 features are enumerated in the doc's "Shipped model" table.
`goalsonly`'s single feature is stated outright too. **`final5swing`'s 8
features are *not* individually listed in the doc** — only rows 0 and 7 get
a full feature list there. This notebook reconstructs it as the shipped
model's 11 minus the 3 team-strength features (`avg_strength`, `gap_strength`,
`upset`), since the doc says row 7 = row 5 + team strength, and row 6 (the
row in between) was tried and rejected. Confident, but worth knowing it's an
inference rather than a literal quote from the doc if these numbers ever need
re-checking.

All three get saved to `../models/`, each self-documenting via
`predict_excitingness.py`'s `build_model_bundle()`. The site's Advanced
selector picks between them by reading whichever `excitingness_<key>`
columns `predict_excitingness.py` produced.

## Setup — import the shipped config from `predict_excitingness.py`

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, "..")
import predict_excitingness as pe

for key, spec in pe.MODEL_SPECS.items():
    print(f"{key:<12} {len(spec['features']):>2} features  -> {spec['path'].name}")

shipped      11 features  -> excitingness_model.joblib
final5swing   8 features  -> excitingness_model_final5swing.joblib
goalsonly     1 features  -> excitingness_model_goalsonly.joblib


## Load training data

Same loader `predict_excitingness.py` itself uses — builds `wc_labelled.csv` from raw shot JSON + IMDb ratings if it doesn't exist yet, then drops extra-time matches.

In [2]:
data_dir = pe.find_data_dir()
pipeline_dir = pe.find_pipeline_dir(data_dir)
print(f"data dir: {data_dir}")

lab = pe.ensure_wc_labels(data_dir, pipeline_dir)
print(f"\ntraining rows: {len(lab)}")

data dir: /Users/gameps/Documents/isdb/data
training matches: 168 -> 154 (extra-time matches excluded)

training rows: 154


## Build the feature matrix

Computed once, unconditionally, for every engineered column — `build_matrix()` doesn't know or care which subset each model actually uses; that selection happens per-model below via `X_wc[spec['features']]`.

In [3]:
wc_cache = {}
for _, r in lab.iterrows():
    p = data_dir / f"xg_timeline/wc/wc{int(r.season_year)}_match_{r.match_id}.json"
    wc_cache[r.match_id] = pe.wc_shots(json.load(open(p))) if p.exists() else []

missing = sum(1 for v in wc_cache.values() if not v)
if missing:
    print(f"warning: {missing}/{len(wc_cache)} training matches have no raw shot file "
          f"on disk \u2014 their shot-derived features will be zero, which will pull "
          f"every model below away from notes/model_history.md's documented values. "
          f"Make sure data/xg_timeline/wc/ is fully populated before trusting this "
          f"run's output.")

wc_maps = {yr: pe.pct_map(d) for yr, d in pe.FIFA_RANK.items()}
X_wc = pe.build_matrix(lab, lambda r: pe.shot_features(wc_cache[r.match_id]), wc_maps)
y = lab.imdb_rating.values
w = (lab.imdb_votes.astype(float) / lab.imdb_votes.mean()).values   # vote-weighted

## Fit and save all three

Same ridge config for every one (`alpha=30`, vote-weighted, standardized) — only the feature subset changes.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import joblib

fitted = {}
for key, spec in pe.MODEL_SPECS.items():
    model = Pipeline([
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=pe.RIDGE_ALPHA)),
    ])
    model.fit(X_wc[spec["features"]].values, y, ridge__sample_weight=w)
    fitted[key] = model

    bundle = pe.build_model_bundle(
        model, len(lab),
        feature_set=spec["features"], key=key, label=spec["label"],
        model_history_row=spec["model_history_row"],
    )
    spec["path"].parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, spec["path"])
    print(f"{key:<12} fitted on {len(lab)} matches, {len(spec['features'])} features "
          f"-> {spec['path']}")

shipped      fitted on 154 matches, 11 features -> /Users/gameps/Documents/isdb/models/excitingness_model.joblib
final5swing  fitted on 154 matches, 8 features -> /Users/gameps/Documents/isdb/models/excitingness_model_final5swing.joblib
goalsonly    fitted on 154 matches, 1 features -> /Users/gameps/Documents/isdb/models/excitingness_model_goalsonly.joblib


## Sanity check the shipped model against `notes/model_history.md`

Only `shipped` gets this check — it's the only alternate whose coefficients the doc actually enumerates row-by-row, so it's the only one where a mismatch is unambiguously diagnosable this way. A coefficient with the **wrong sign**, or one that collapsed to ~0 when the doc shows it as a real contributor, means something's missing upstream (usually incomplete raw shot files) — not that the doc is wrong.

In [5]:
documented = {
    "total_goals": 0.395, "avg_strength": -0.352, "upset": 0.202,
    "chasing_xg": 0.200, "big_chances_60-75": 0.180, "final5_swing_count": 0.145,
    "xg_absdiff_30-45": 0.117, "xg_absdiff_75-90plus": -0.060,
    "xg_absdiff_0-15": 0.048, "gap_strength": -0.039, "goal_diff_abs": -0.034,
}

ridge = fitted["shipped"].named_steps["ridge"]
this_run = dict(zip(pe.MODEL_SPECS["shipped"]["features"], ridge.coef_.round(3)))

print(f"{'feature':<22}{'this run':>10}{'documented':>12}{'sign match':>12}")
for feat in pe.MODEL_SPECS["shipped"]["features"]:
    a, b = this_run[feat], documented.get(feat, float('nan'))
    sign_ok = "OK" if (a * b) > 0 or abs(a) < 0.02 else "!! CHECK"
    print(f"{feat:<22}{a:>10.3f}{b:>12.3f}{sign_ok:>12}")

feature                 this run  documented  sign match
total_goals                0.395       0.395          OK
goal_diff_abs             -0.034      -0.034          OK
xg_absdiff_30-45           0.117       0.117          OK
xg_absdiff_0-15            0.048       0.048          OK
chasing_xg                 0.199       0.200          OK
final5_swing_count         0.145       0.145          OK
big_chances_60-75          0.180       0.180          OK
xg_absdiff_75-90plus      -0.060      -0.060          OK
avg_strength              -0.352      -0.352          OK
gap_strength              -0.039      -0.039          OK
upset                      0.202       0.202          OK


## Preview the other two

No documented table to check these against, but worth eyeballing that the signs at least make intuitive sense (more goals → more exciting; being further behind in the final 5 minutes with the swing count feature present → more exciting, etc.).

In [6]:
for key in ("final5swing", "goalsonly"):
    ridge = fitted[key].named_steps["ridge"]
    print(f"\n{key}:")
    for feat, coef in zip(pe.MODEL_SPECS[key]["features"], ridge.coef_.round(3)):
        print(f"  {feat:<22}{coef:>8.3f}")


final5swing:
  total_goals              0.415
  goal_diff_abs           -0.130
  xg_absdiff_30-45         0.115
  xg_absdiff_0-15          0.122
  chasing_xg               0.215
  final5_swing_count       0.184
  big_chances_60-75        0.171
  xg_absdiff_75-90plus    -0.079

goalsonly:
  total_goals              0.492


## Use it

```bash
python3 predict_excitingness.py --fetch
```

With all three files present, this now writes `excitingness` (shipped, the default) plus `excitingness_final5swing` and `excitingness_goalsonly` — the columns `build_pl_frontend.py` picks up automatically for the site's Advanced selector. Re-run this notebook only when the shipped feature set or training data actually changes, not on every scoring run.